# Quiz DL 1 - Klasifikasi Sayur dengan DCNN
Broccoli, Carrot, Cucumber, Tomato | grayscale 224x224 | split 80/10/10

Alur: Import -> Load -> Split -> Preprocess -> Model -> Train -> Predict & Evaluate -> Plot

## 1. Import Library

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import keras
from sklearn.model_selection import train_test_split

## 2. Load Dataset

In [2]:
PATH = "./Dataset/"
CLASS_NAMES = sorted(os.listdir(PATH))   # sorted -> urutan label pasti (alfabet)
print("Classes:", CLASS_NAMES)

Classes: ['Broccoli', 'Carrot', 'Cucumber', 'Tomato']


In [3]:
rows, cols = 224, 224
x, y = [], []

for label, class_name in enumerate(CLASS_NAMES):          # label: 0, 1, 2, 3
    for img_name in os.listdir(PATH + class_name):
        img = keras.utils.load_img(f"{PATH}{class_name}/{img_name}",
                                   target_size=(rows, cols), color_mode="grayscale")
        x.append(keras.utils.img_to_array(img))           # (224, 224, 1)
        y.append(label)

x, y = np.array(x), np.array(y)

# Random shuffle: satu urutan acak dipakai untuk x dan y agar tetap berpasangan
idx = np.random.permutation(len(x))
x, y = x[idx], y[idx]

print(x.shape, y.shape)
print(np.unique(y, return_counts=True))

(2000, 224, 224, 1) (2000,)
(array([0, 1, 2, 3]), array([500, 500, 500, 500]))


## 3. Split Dataset (80 / 10 / 10)

In [4]:
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(len(x_train), len(x_val), len(x_test))

1600 200 200


## 4. Data Preprocessing

In [5]:
# 1. reshape (jumlah gambar, tinggi, lebar, channel) | 2. float32 | 3. normalisasi /255
x_train = x_train.reshape(-1, rows, cols, 1).astype("float32") / 255.0
x_val   = x_val.reshape(-1, rows, cols, 1).astype("float32") / 255.0
x_test  = x_test.reshape(-1, rows, cols, 1).astype("float32") / 255.0

# 4. one-hot encoding label
NUM_CLASSES = len(CLASS_NAMES)
y_train = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_val   = keras.utils.to_categorical(y_val, NUM_CLASSES)
y_test  = keras.utils.to_categorical(y_test, NUM_CLASSES)

## 5. Arsitektur DCNN

In [ ]:
model = keras.Sequential([
    # Input layer
    keras.layers.Input(shape=(rows, cols, 1)),

    # Hidden layer: Conv -> Pool -> Conv -> Conv -> Pool -> Flatten
    keras.layers.Conv2D(6, (5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    keras.layers.Conv2D(16, (5, 5), activation="relu"),
    keras.layers.Conv2D(16, (5, 5), activation="relu"),
    keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
    keras.layers.Flatten(),

    # Output layer: Dense ReLU -> Dense Softmax
    keras.layers.Dense(120, activation="relu"),
    keras.layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.summary()

## 6. Training

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(x_train, y_train, batch_size=32, epochs=10,
                    validation_data=(x_val, y_val))

## 7. Predict & Evaluate

In [ ]:
# Predict: probabilitas per kelas -> ambil kelas terbesar
pred = model.predict(x_test)
pred_class = np.argmax(pred, axis=1)
true_class = np.argmax(y_test, axis=1)

pred_loss = keras.losses.categorical_crossentropy(y_test, pred).numpy().mean()
pred_acc = (pred_class == true_class).mean()
print(f"[Predict]  Loss: {pred_loss:.4f} | Accuracy: {pred_acc:.4f}")

In [ ]:
# Evaluate: model.evaluate mengembalikan [loss, accuracy]
loss, acc = model.evaluate(x_test, y_test)
print(f"[Evaluate] Loss: {loss:.4f} | Accuracy: {acc:.4f}")

## 8. Plot Evaluation Metrics

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()

plt.tight_layout()
plt.show()